# Camada Silver: pesquisa_satisfacao

Trata a tabela `bronze.pesquisa_satisfacao` e grava `silver.tb_pesquisa_satisfacao`. Converte a nota para inteiro, anula notas faltantes e fora da escala de 1 a 5 e deixa uma pesquisa por chamado. A validação de integridade lê `silver.tb_chamados`, então a Silver de chamados precisa rodar antes. As análises que embasam esses tratamentos estão em `Bronze_Analysis_pesquisa_satisfacao`.

In [0]:
from pyspark.sql import functions as F, Window

# criação dos nomes/caminhos dos schemas de cada camada
catalog = "vcredit_medallion"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")

## Funções auxiliares

In [0]:
TOKENS_NULOS = ["NULL", "NONE", "NAN", "N/A"]      # textos que na prática significam "sem valor"


def texto(c):
    """Coluna como string, sem espaços nas pontas."""
    return F.trim(F.col(c).cast("string"))


def eh_faltante(c):
    """Null de verdade, string vazia/só espaços, ou texto tipo 'NULL'."""
    return F.col(c).isNull() | (texto(c) == "") | F.upper(texto(c)).isin(TOKENS_NULOS)


def somente_digitos(c):
    """Mantém só 0-9, como string (mesmo padrão dos ids da tb_chamados). Sem nenhum dígito -> null."""
    d = F.regexp_replace(F.col(c).cast("string"), r"[^0-9]", "")
    return F.when(d == "", F.lit(None).cast("string")).otherwise(d)


def ultima_versao(df, chaves):
    """Fica com a carga mais recente de cada chave (a Bronze é append) e descarta ingestion_datetime.
    Também elimina cópias exatas da mesma chave dentro de uma carga."""
    w = Window.partitionBy(*chaves).orderBy(F.col("ingestion_datetime").desc())
    return df.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn", "ingestion_datetime")


def checar(regra, df_falhas):
    """Para o notebook se alguma linha violar a regra."""
    n = df_falhas.count()
    assert n == 0, f"FALHA na validação '{regra}': {n} linha(s)"
    print(f"OK: {regra}")


def salvar(df, nome):
    (df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{silver_schema}.{nome}"))
    print(f"{silver_schema}.{nome}: {spark.table(f'{silver_schema}.{nome}').count()} linhas")

## 1. Leitura da Bronze

In [0]:
b_pesquisa = spark.table(f"{bronze_schema}.pesquisa_satisfacao")
total_bronze = b_pesquisa.count()
print(f"Linhas na Bronze: {total_bronze}")
display(b_pesquisa)

## 2. Ids e última versão
`ultima_versao` por `id_pesquisa` remove cargas repetidas. Na Bronze não havia cópias exatas.

In [0]:
pesquisa = ultima_versao(
    b_pesquisa.withColumn("id_pesquisa", somente_digitos("id_pesquisa"))
              .withColumn("id_chamado", somente_digitos("id_chamado")),
    ["id_pesquisa"],
)
print(f"Linhas após última versão: {pesquisa.count()}")

## 3. Conversão da nota
Faltantes (`NULL`, vazio) e notas fora da escala de 1 a 5 (-1, 0, 6) viram null.

In [0]:
n = texto("nota_atendimento")
pesquisa = pesquisa.withColumn("_nota_int", F.when(n.rlike(r"^-?\d+$"), n.cast("int")))

checar("nota: todo valor preenchido na Bronze é um inteiro",
       pesquisa.filter(~eh_faltante("nota_atendimento") & F.col("_nota_int").isNull()))

n_faltantes = pesquisa.filter(eh_faltante("nota_atendimento")).count()
n_fora_escala = pesquisa.filter(~F.col("_nota_int").between(1, 5)).count()
print(f"Notas faltantes convertidas para null: {n_faltantes}")
print(f"Notas fora da escala convertidas para null: {n_fora_escala}")

pesquisa = (
    pesquisa
    .withColumn("nota_atendimento", F.when(F.col("_nota_int").between(1, 5), F.col("_nota_int")))
    .drop("_nota_int")
)

## 4. Pesquisa duplicada por chamado
Na Bronze, 802 chamados tinham 2 pesquisas com a mesma nota e outro `id_pesquisa`. Fica a de menor `id_pesquisa`.
A validação abaixo confirma que as duplicatas têm a mesma nota antes de descartar uma delas.

In [0]:
checar("pesquisas do mesmo chamado têm a mesma nota",
       pesquisa.groupBy("id_chamado")
               .agg(F.size(F.collect_set(F.coalesce(F.col("nota_atendimento"), F.lit(-1)))).alias("notas"))
               .filter("notas > 1"))

w = Window.partitionBy("id_chamado").orderBy(F.col("id_pesquisa").cast("long"))
antes = pesquisa.count()
pesquisa = (
    pesquisa
    .withColumn("_rn", F.row_number().over(w))
    .filter("_rn = 1")
    .drop("_rn")
    .select("id_pesquisa", "id_chamado", "nota_atendimento")
    .orderBy(F.col("id_pesquisa").cast("long"))
)
print(f"Pesquisas duplicadas removidas: {antes - pesquisa.count()}")
display(pesquisa)

## 5. Validações
A checagem de integridade usa a `silver.tb_chamados`, então a Silver de chamados precisa rodar antes deste notebook.

In [0]:
checar("pesquisa: id_pesquisa e id_chamado preenchidos",
       pesquisa.filter(F.col("id_pesquisa").isNull() | F.col("id_chamado").isNull()))
checar("pesquisa: id_pesquisa único", pesquisa.groupBy("id_pesquisa").count().filter("count > 1"))
checar("pesquisa: uma pesquisa por chamado", pesquisa.groupBy("id_chamado").count().filter("count > 1"))
checar("pesquisa: nota nula ou entre 1 e 5",
       pesquisa.filter(F.col("nota_atendimento").isNotNull() & ~F.col("nota_atendimento").between(1, 5)))

ids_chamados = spark.table(f"{silver_schema}.tb_chamados").select("id_chamado").distinct()
checar("pesquisa: todo id_chamado existe na tb_chamados",
       pesquisa.join(ids_chamados, on="id_chamado", how="left_anti"))

## 6. Gravação da Silver
Só chega aqui se todas as validações passaram.

In [0]:
salvar(pesquisa, "tb_pesquisa_satisfacao")